In [90]:
import json
import os
import sys
import re
import unicodedata
import pandas as pd

In [91]:
# Readin
df = pd.read_csv('MedSynth_huggingface_final.csv')
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:**\n\n **Chief Complaint (CC...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:**\n\n - **Chief Complaint (...,"[doctor] Hi there, how are you today?\n\n[pati...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:**\n\n**Chief Complaint (CC):*...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:**\n\n**Chief Complaint (CC):*...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,#####\n**1. Subjective:**\n\n**Chief Complaint...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,#####\n**1. Subjective:**\n \n**Chief Compla...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,### Gastroenterologist Medical Note\n\n#### 1....,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:**\n\n**Chief Complaint (CC):*...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,#####\n**1. Subjective:**\n**Chief Complaint (...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [92]:
# Harmonizing UTF characters
def clean_string(s):
    if not isinstance(s, str):
        return s

    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)
    s = re.sub(r"[\x00-\x1F\x7F]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df[["Note", "Dialogue"]] = df[["Note", "Dialogue"]].applymap(clean_string)

C:\Users\Keno\AppData\Local\Temp\ipykernel_17716\3193131587.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[["Note", "Dialogue"]] = df[["Note", "Dialogue"]].applymap(clean_string)


In [93]:
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:** **Chief Complaint (CC):** -...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:** - **Chief Complaint (CC):**...,"[doctor] Hi there, how are you today?[patient]...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:****Chief Complaint (CC):** Mo...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,#####**1. Subjective:****Chief Complaint (CC):...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,#####**1. Subjective:** **Chief Complaint (CC)...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,### Gastroenterologist Medical Note#### 1. Sub...,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:****Chief Complaint (CC):**Dif...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,#####**1. Subjective:****Chief Complaint (CC):...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [94]:
"""df = df.apply(
    lambda col: col.str.replace(r"\n{2,}", "\n", regex=True)
    if col.dtype == "object" else col
)"""

'df = df.apply(\n    lambda col: col.str.replace(r"\n{2,}", "\n", regex=True)\n    if col.dtype == "object" else col\n)'

In [95]:
# Erasing leading styling characters

s = df["Note"].astype("string")
df["Note"] = s.apply(lambda x: x[x.find("**"):] if isinstance(x, str) and "**" in x else x)

In [96]:
df

,Note,Dialogue,ICD10,ICD10_desc
0,**1. Subjective:** **Chief Complaint (CC):** -...,[doctor]: Hello! It’s good to see you today. H...,M25562,PAIN IN LEFT KNEE
1,**1. Subjective:** - **Chief Complaint (CC):**...,"[doctor] Hi there, how are you today?[patient]...",M25562,PAIN IN LEFT KNEE
2,**1. Subjective:****Chief Complaint (CC):**Sev...,"[doctor] Good morning, how are you doing today...",M25562,PAIN IN LEFT KNEE
3,**1. Subjective:****Chief Complaint (CC):** Mo...,[doctor] Good morning! How are you feeling tod...,M25562,PAIN IN LEFT KNEE
4,**1. Subjective:****Chief Complaint (CC):** Mo...,"[doctor]: Hello Mr. Doe, how are you doing tod...",M25562,PAIN IN LEFT KNEE
...,...,...,...,...
10235,**1. Subjective:** **Chief Complaint (CC):**Di...,[doctor]: Good morning. How are you doing toda...,B3781,CANDIDAL ESOPHAGITIS
10236,**Chief Complaint (CC):** Difficulty swallowin...,"**Doctor:** Hi there, how are you doing today?...",B3781,CANDIDAL ESOPHAGITIS
10237,**1. Subjective:****Chief Complaint (CC):**Dif...,"[doctor]: Hi Mr. Harris, how are you doing tod...",B3781,CANDIDAL ESOPHAGITIS
10238,**1. Subjective:****Chief Complaint (CC):** Se...,"[doctor]: Good morning, Ms. Lee. How are you d...",B3781,CANDIDAL ESOPHAGITIS


In [ ]:
# Drop NA (and one abnormative) values

df.isna().sum()
df = df.drop([10236])
df = df.dropna()
df.reset_index
df

Note          0
Dialogue      2
ICD10         0
ICD10_desc    0
dtype: int64

In [ ]:
df.to_json("finetuning_data.json", orient="records")